In [ ]:
#CELL 1 — Install Required Libraries
!pip install sentence-transformers transformers accelerate gradio gtts sacrebleu datasets matplotlib seaborn

INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
  Attempting uninstall: typer
    Found existing installation: typer 0.24.1
    Uninstalling typer-0.24.1:
      Successfully uninstalled typer-0.24.1
  Attempting uninstall: typer-slim
    Found existing installation: typer-slim 0.24.0
    Uninstalling typer-slim-0.24.0:
      Successfully uninstalled typer-slim-0.24.0


In [ ]:
#CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#CELL 3 — Load Dataset
import json

base_path = "/content/drive/MyDrive/majorproject/"

with open(base_path + "bibledata/kjv.json") as f:
    bible = json.load(f)

with open(base_path + "gitadata/verse.json") as f:
    gita = json.load(f)

with open(base_path + "qurandata/quran_en.json") as f:
    quran = json.load(f)


# 🔵 Bible
def prepare_bible(data):
    verses = []
    for item in data["verses"]:
        verses.append({
            "text": item["text"],
            "book": item["book_name"],
            "chapter": item["chapter"],
            "verse": item["verse"],
            "source": "Bible"
        })
    return verses


# 🔵 Gita
def prepare_gita(data):
    verses = []
    for item in data:
        gita_text = item.get("transliteration", "").strip()
        meaning = item.get("word_meanings", "").strip()

        combined_text = gita_text
        if meaning:
            combined_text = combined_text + " " + meaning

        verses.append({
            "text": combined_text,
            "book": "Bhagavad Gita",
            "chapter": item["chapter_number"],
            "verse": item["verse_number"],
            "source": "Gita"
        })
    return verses


# 🔵 Quran
def prepare_quran(data):
    verses = []
    for surah in data:
        for verse in surah["verses"]:
            verses.append({
                "text": verse["translation"],   # English
                "book": surah["transliteration"],
                "chapter": surah["id"],
                "verse": verse["id"],
                "source": "Quran"
            })
    return verses


all_data = (
    prepare_bible(bible) +
    prepare_gita(gita) +
    prepare_quran(quran)
)

print("Total verses:", len(all_data))
print(all_data[0])

Total verses: 38039
{'text': '¶ In the beginning God created the heaven and the earth.', 'book': 'Genesis', 'chapter': 1, 'verse': 1, 'source': 'Bible'}


In [ ]:
#CELL 4 — Load BGE Model (Retrieval)
from sentence_transformers import SentenceTransformer

model_bge = SentenceTransformer("BAAI/bge-base-en")



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 13.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.35 GiB is allocated by PyTorch, and 77.02 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
all_data = (
    prepare_bible(bible) +
    prepare_gita(gita) +
    prepare_quran(quran)
)

texts = [v["text"] for v in all_data]

embeddings = model_bge.encode(
    texts,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

In [ ]:
#CELL 5 — Retrieval Function + METRICS
from sentence_transformers import util

def retrieve(query, scripture="all", top_k=3):
    query_emb = model_bge.encode(query, convert_to_tensor=True, normalize_embeddings=True)

    scores = util.cos_sim(query_emb, embeddings)[0]

    filtered = []
    for i, v in enumerate(all_data):
        if scripture.lower() == "all" or v["source"].lower() == scripture.lower():
            filtered.append((i, scores[i].item()))

    filtered = sorted(filtered, key=lambda x: x[1], reverse=True)[:top_k]

    results = [all_data[i] for i, _ in filtered]
    sim_scores = [s for _, s in filtered]

    return results, sim_scores


# ✅ REAL METRICS (no hallucination)
def retrieval_metrics(scores):
    return {
        "avg_similarity": sum(scores)/len(scores),
        "max_similarity": max(scores),
        "min_similarity": min(scores)
    }

In [ ]:
#CELL 6 — Test Retrieval
query = "I feel lost in life"

results, scores = retrieve(query)

for r, s in zip(results, scores):
    print(f"{r['source']} | {r['book']} {r['chapter']}:{r['verse']}")
    print(r["text"])
    print("Score:", s)
    print("-"*50)

print("Metrics:", retrieval_metrics(scores))

Bible | Psalms 102:11
My days [are] like a shadow that declineth; and I am withered like grass.
Score: 0.8191922903060913
--------------------------------------------------
Bible | Psalms 31:12
I am forgotten as a dead man out of mind: I am like a broken vessel.
Score: 0.8176877498626709
--------------------------------------------------
Quran | Al-Haqqah 69:28
My wealth has not availed me
Score: 0.8175097703933716
--------------------------------------------------
Metrics: {'avg_similarity': 0.8181299368540446, 'max_similarity': 0.8191922903060913, 'min_similarity': 0.8175097703933716}


In [ ]:
#CELL 7 — Phi-3 Mini (Explanation Model)
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

phi_model_name = "microsoft/phi-3-mini-4k-instruct"

tokenizer_phi = AutoTokenizer.from_pretrained(phi_model_name)
model_phi = AutoModelForCausalLM.from_pretrained(
    phi_model_name,
    device_map="auto",
    torch_dtype="auto"
)

def generate_explanation(query, verses):
    verses_text = "\n".join(
        [f"{v['source']} | {v['book']} {v['chapter']}:{v['verse']} - {v['text']}" for v in verses]
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a compassionate spiritual guide. "
                "Write exactly one paragraph only. "
                "Start with 'Dear friend,'. "
                "Use only the provided verses. "
                "Do not repeat the prompt. "
                "Do not print labels, headings, bullet points, or instructions."
            )
        },
        {
            "role": "user",
            "content": (
                f"User problem: {query}\n\n"
                f"Retrieved verses:\n{verses_text}\n\n"
                "Write one compassionate paragraph."
            )
        }
    ]

    prompt = tokenizer_phi.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer_phi(prompt, return_tensors="pt")
    inputs = {k: v.to(model_phi.device) for k, v in inputs.items()}

    outputs = model_phi.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False,
        temperature=None,
        top_p=None,
        eos_token_id=tokenizer_phi.eos_token_id
    )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    explanation = tokenizer_phi.decode(generated, skip_special_tokens=True).strip()

    if "Dear friend" in explanation:
        explanation = explanation[explanation.index("Dear friend"):].strip()

    explanation = " ".join(explanation.split())
    return explanation

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [ ]:
#CELL 8 — Explanation Metrics (REAL)
from sentence_transformers import util

def explanation_metrics(explanation, verses):
    verse_text = " ".join([v["text"] for v in verses])

    emb_expl = model_bge.encode(explanation, convert_to_tensor=True, normalize_embeddings=True)
    emb_verses = model_bge.encode(verse_text, convert_to_tensor=True, normalize_embeddings=True)

    semantic_similarity = util.cos_sim(emb_expl, emb_verses).item()

    return {
        "semantic_similarity": semantic_similarity,
        "word_count": len(explanation.split()),
        "char_count": len(explanation)
    }

In [ ]:
#CELL 10 — NLLB Translation
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer_nllb = AutoTokenizer.from_pretrained(model_name)
model_nllb = AutoModelForSeq2SeqLM.from_pretrained(model_name, device_map="auto")


def translate(text, lang):
    lang_map = {
        "english": "eng_Latn",
        "telugu": "tel_Telu",
        "hindi": "hin_Deva",
        "bengali": "ben_Beng"
    }

    target_lang = lang_map[lang]

    inputs = tokenizer_nllb(text, return_tensors="pt").to("cuda")

    translated_tokens = model_nllb.generate(
        **inputs,
        forced_bos_token_id=tokenizer_nllb.convert_tokens_to_ids(target_lang),
        max_length=512
    )

    return tokenizer_nllb.decode(translated_tokens[0], skip_special_tokens=True)

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
#CELL 11 — Translation Metrics (REAL BLEU)
import sacrebleu

def translation_metrics(original, translated):
    bleu = sacrebleu.corpus_bleu([translated], [[original]])

    return {
        "bleu_score": bleu.score
    }

In [ ]:
#CELL 12 — Full Pipeline Test
query = "I am feeling stressed and hopeless"

verses, scores = retrieve(query)

explanation = generate_explanation(query, verses)

translation = translate(explanation, "hindi")

print("Explanation:\n", explanation)
print("\nTranslation:\n", translation)

print("\nRetrieval Metrics:", retrieval_metrics(scores))
print("Explanation Metrics:", explanation_metrics(explanation))
print("Translation Metrics:", translation_metrics(explanation, translation))

KeyboardInterrupt: 

In [ ]:
#CELL 10 — Text to Speech
from gtts import gTTS
import os

def text_to_audio(text, filename):
    tts = gTTS(text)
    tts.save(filename)
    return filename

In [ ]:
#CELL 11 — GRADIO UI (Gold Theme)
import gradio as gr

def pipeline(query, scripture, lang):
    verses, scores = retrieve(query, scripture)

    explanation = generate_explanation(query, verses)
    translation = translate(explanation, lang)

    verse_text = "\n".join([f"{v['text']} ({v['reference']})" for v in verses])

    v_audio = text_to_audio(verse_text, "verse.mp3")
    e_audio = text_to_audio(explanation, "explanation.mp3")
    t_audio = text_to_audio(translation, "translation.mp3")

    return verse_text, v_audio, explanation, e_audio, translation, t_audio

iface = gr.Interface(
    fn=pipeline,
    inputs=[
        gr.Textbox(label="Enter your problem"),
        gr.Dropdown(["all","bible","gita","quran"]),
        gr.Dropdown(["english","telugu","hindi","bengali"])
    ],
    outputs=[
        gr.Textbox(label="Verses"),
        gr.Audio(),
        gr.Textbox(label="Explanation"),
        gr.Audio(),
        gr.Textbox(label="Translation"),
        gr.Audio()
    ],
    title="✨ TrinityVerseAI ✨",
    theme=gr.themes.Default(primary_hue="yellow")
)

iface.launch()

In [ ]:
#CELL 12 — METRICS + GRAPH VISUALIZATION
import matplotlib.pyplot as plt

def evaluate_pipeline(query):
    verses, scores = retrieve(query)
    explanation = generate_explanation(query, verses)
    translation = translate(explanation, "hindi")

    r_metrics = retrieval_metrics(scores)
    e_metrics = explanation_metrics(explanation)
    t_metrics = translation_metrics(explanation, translation)

    print("Retrieval:", r_metrics)
    print("Explanation:", e_metrics)
    print("Translation:", t_metrics)

    labels = ["Similarity", "Readability", "BLEU"]
    values = [
        r_metrics["avg_similarity"],
        e_metrics["readability_proxy"],
        t_metrics["bleu_score"]
    ]

    plt.figure()
    plt.bar(labels, values)
    plt.title("Model Performance")
    plt.show()

# Example
evaluate_pipeline("I am feeling lost in life")